**강사: 멀티캠퍼스 강선구 강사** (sun9sun9@gmail.com)

In [1]:
import os
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text
import cx_Oracle
from IPython.core.magic import register_cell_magic
import sys
if sys.version.find('GCC') >= 0:
    domain_name = 'oracle-db'
else:
    domain_name = 'localhost'
    cx_Oracle.init_oracle_client(lib_dir = r"C:\Oracle\instantclient_23_7")
DATABASE_URL = "oracle+cx_oracle://lib:multisqld@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)
def get_rows(qry):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(qry))
            df = pd.DataFrame(result.fetchall(), columns = result.keys()).fillna('NULL').pipe(lambda x: x.set_index(x.index + 1))
            return df if len(df) != 1 else df.iloc[0]
    except sqlalchemy.exc.DatabaseError as e:
        orig = e.orig
        offset = getattr(orig, 'offset', None)
        message = getattr(orig, 'message', str(orig))
        print(message, offset)
    except Exception as e:
        print(e.__class__)
        print(e)

def exec_qrys(qrys):
    try:
        results = list()
        with engine.connect() as conn:
            for qry in qrys.split(';'):
                qry = qry.strip()
                if len(qry) == 0:
                    continue
                result = conn.execute(text(qry))
                results.append(
                    "{}, 변경된 행의 수: {}".format(qry, result.rowcount)
                )
        return results
    except Exception as e:
        print(e)

def exec_qry(qry):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(qry))
            conn.commit()
            return "변경된 행의 수: {}".format(result.rowcount)
    except Exception as e:
        print(e)

@register_cell_magic
def SQL(line, cell):
    sql = cell.strip().replace(';', '')
    if sql.lower().startswith('select'):
        return get_rows(sql)
    elif sql.lower().startswith('desc'):
        return get_rows(
"""
SELECT column_name, data_type, data_length, nullable
FROM user_tab_columns
WHERE table_name = '{}'
ORDER BY column_id
""".format(sql[5:].upper().strip())
        )
        
    else:
        exec_qry(sql)

# 서브쿼리 

**동작 방식에 따른 분류**
|종류|설명|
|------|-----|
|비연관(Un-Correlated) 서브 쿼리|메인 쿼리의 컬럼을 참조하지 않는 서브 쿼리.|
|연관(Correlated)서브 쿼리|메인 쿼리의 컬럼을 참조하는 서브 쿼리.<br/>EXISTS (연관 서브쿼리) 연산<br/>EXISTS의 연관 서브 쿼리가 반환하는 행이 하나 이상이면 참(TRUE)을 반환|


**[예제 1]** 서브 쿼리의 종류를 살펴봅니다.

In [2]:
%%SQL

SELECT * FROM member1 -- member1의 내용을 봅니다.

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


- 비연관 서브 쿼리 입니다. in 연산을 통해 member1에서 가져온 member_id를 포함된 rating의 목록을 가져옵니다.

In [3]:
%%SQL

SELECT * FROM rating r
WHERE member_id in (
    SELECT member_id FROM member1
)

,book_id,member_id,rating,rate_date
1,10,zcitby9bvw,8,2019-11-04 12:57:34
2,48,9005r5vmi,10,2019-11-10 11:16:46
3,48,zcitby9bvw,9,2021-11-11 09:23:38
4,100,nnvy9vtclgyfb,8,2020-08-08 12:01:15
5,150,osfi8te4nv9m,9,2021-10-13 09:41:52
6,312,9005r5vmi,9,2019-10-29 10:10:06
7,325,q6bl2tdplxz,9,2021-12-12 12:32:10
8,391,nnvy9vtclgyfb,10,2023-11-28 17:00:21
9,780,9005r5vmi,10,2023-11-19 10:43:14
10,893,kijtmcsxqvxe4d,7,2024-04-22 16:42:36


- 연관 서브 쿼리 입니다. exists 절에서는 FROM 절에서 가져온 r을 참조하여 member1에서 있을 경우 1이 반환되어 가져와지도록 합니다.

In [4]:
%%SQL

SELECT * 
FROM rating r
WHERE exists (
    SELECT 1 FROM member1 m WHERE m.member_id = r.member_id
)

,book_id,member_id,rating,rate_date
1,10,zcitby9bvw,8,2019-11-04 12:57:34
2,48,9005r5vmi,10,2019-11-10 11:16:46
3,48,zcitby9bvw,9,2021-11-11 09:23:38
4,100,nnvy9vtclgyfb,8,2020-08-08 12:01:15
5,150,osfi8te4nv9m,9,2021-10-13 09:41:52
6,312,9005r5vmi,9,2019-10-29 10:10:06
7,325,q6bl2tdplxz,9,2021-12-12 12:32:10
8,391,nnvy9vtclgyfb,10,2023-11-28 17:00:21
9,780,9005r5vmi,10,2023-11-19 10:43:14
10,893,kijtmcsxqvxe4d,7,2024-04-22 16:42:36


**반환되는 데이터의 수에 따른 분류**

|종류|설명|
|----|----|
|단일행(Single Row) 서브 쿼리|항상 1건 이하의 행을 반환|
|다중행(Multi Row) 서브 쿼리| 1건이 넘는 행을 반환|
|다중컬럼(Multi Column) 서브 쿼리| 여러 컬럼을 반환|
|인라인 뷰(Inline view| 테이블을 반환

**[예제 2]** 반환되는 데이터의 수에 따른 분류에 따른 서브 쿼리를 살펴봅니다.

In [5]:
%%SQL

SELECT avg(rating) FROM rating -- rating에서 평균을 구합니다.

AVG(RATING)    8.57606088739313759622451742679200030947
Name: 1, dtype: object

- 조건문에서 rating의 평균보다 큰 데이터 수를 구합니다. : 단일행 서브쿼리

In [6]:
%%SQL

SELECT count(*)
FROM rating
WHERE rating > (SELECT avg(rating) FROM rating)

COUNT(*)    63753
Name: 1, dtype: int64

- 선택 표현식 내부에서 서브 쿼리를 사용합니다. rating에서 member_id, book_id, rating과 함께,

  평균이상 이면 '평균이상' 그렇지 않으면 '평균미만'을 출력하는 컬럼을 만듭니다. : 단일행 서브 쿼리

In [7]:
%%SQL

SELECT member_id, book_id, rating,
    CASE
        WHEN rating >= (SELECT avg(rating) FROM rating) THEN '평균이상'
        ELSE '평균미만'
    END 평점구분
FROM rating

,member_id,book_id,rating,평점구분
1,g9h5abtvl4py6c3z4,6,10,평균이상
2,go5lwnkx36ay,6,9,평균이상
3,grzfpmuao1b0,6,8,평균미만
4,htkhgeto9kdh,6,8,평균미만
5,iizxw855kaut3,6,9,평균이상
...,...,...,...,...
103400,vo3x1h9uptzqj,3033,7,평균미만
103401,vrk43uoz610a7,3033,10,평균이상
103402,ww5qd8b0bnfz,3033,10,평균이상
103403,x3ionc0swm,3033,8,평균미만


In [8]:
%%SQL

SELECT * FROM book1 -- book1의 내용을 봅니다.

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


In [9]:
%%SQL

SELECT publisher, min(pub_date)
FROM book1
GROUP BY publisher -- book1에서 publisher 별 pub_date의 최소값을 가져옵니다.

,publisher,MIN(PUB_DATE)
1,풀빛,2004-01-01
2,책읽는고양이,2004-07-01
3,문학사상,2000-10-01
4,군자출판사,2023-02-01
5,꿈을담는틀,2024-10-01


- book1에서 publisher별 pub_date의 최소값에 해당하는 publisher와 pub_date를 가져옵니다: 다중 컬럼 / 다중행

In [10]:
%%SQL

SELECT * 
FROM book1
WHERE (publisher, pub_date) in (
    SELECT publisher, min(pub_date)
    FROM book1
    GROUP BY publisher
)

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
5,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- rent1과 rating에서 book_id 별 평균을 구하여 book_id 기준으로 결합을 합니다.

In [11]:
%%SQL
SELECT book_id, avg(rating) as book_avg_rating
FROM rating
GROUP BY book_id

,book_id,book_avg_rating
1,6,8.82716049382716049382716049382716049383
2,14,8.32
3,23,8.78964401294498381877022653721682847896
4,27,9.2
5,50,9.625
...,...,...
1681,3280,8.80459770114942528735632183908045977011
1682,2808,1
1683,2819,8.88636363636363636363636363636363636364
1684,3021,1.33333333333333333333333333333333333333


In [12]:
%%SQL

SELECT a.*, round(b.book_avg_rating, 2) as 도서평점
FROM rent1 a LEFT OUTER JOIN (
    SELECT book_id, avg(rating) as book_avg_rating
    FROM rating
    GROUP BY book_id
) b ON a.book_id = b.book_id

,book_id,stock_seq,member_id,rent_no,rent_date,return_date,도서평점
1,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL,8.14
2,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL,10
3,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23,8.5
4,3194,3,94zfx2tf92heh59z,1,NULL,NULL,8.98
5,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL,8.08
6,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL,8.94
7,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44,9.15
8,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21,8.84
9,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32,9.06
10,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36,8.6


# 집합 연산자

|종류|설명|
|----|-----|
|UNION|중복된 행은 하나의 행으로 하는 합집합 연산|
|UNION ALL|중복 여부와 상관없이 모든 행들을 포함하는 합집합 연산|
|INTERSECT|교집합 연산. 중복된 행은 하나의 행으로 통합됨|
|EXCEPT	차집합 연산|중복된 행은 제외가 됨. (Oracle :MINUS 연산자, SQL Server: EXCEPT)|


[**예제 3**] 집합 연산자의 기능을 알아봅니다.

In [13]:
%%SQL

SELECT * FROM member1

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


member1은 member의 부분 집합니다. 3번째 컬럼의 member_id 보다 같거나 큰 행들을 가져와 member1의 내용을 가진 행을 가져옵니다,

In [14]:
%%SQL

SELECT member_id, member_name, region, address1, address2 
FROM member 
WHERE member_id >= '3huv3i06r0kjq' 
ORDER BY member_id
OFFSET 0 ROWS FETCH FIRST 3 ROWS ONLY

,member_id,member_name,region,address1,address2
1,3huwq6vwehb7,김하은,경상북도,포항시 북구,성곡서로 5번길
2,3hvd0w7gxl2a1o40,전이현,충청남도,아산시,영인로 5번길
3,3hxi8nfz5h,김서준,경상남도,합천군,핫들 8로


- 위에서 가져온 두 개의 결과를 UNION 으로 결합니다. 

In [15]:
%%SQL

SELECT * FROM member1
UNION
(
    SELECT member_id, member_name, region, address1, address2 
    FROM member 
    WHERE member_id >= '3huv3i06r0kjq'
    ORDER BY member_id
    OFFSET 0 ROWS FETCH FIRST 3 ROWS ONLY
)

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길
9,3huwq6vwehb7,김하은,경상북도,포항시 북구,성곡서로 5번길
10,3hvd0w7gxl2a1o40,전이현,충청남도,아산시,영인로 5번길


- 위에서 가져온 두 개의 결과를 UNION ALL 결합니다. 

In [16]:
%%SQL

SELECT * FROM member1
UNION ALL
(
    SELECT member_id, member_name, region, address1, address2 
    FROM member 
    WHERE member_id >= '3huv3i06r0kjq'
    ORDER BY member_id
    OFFSET 0 ROWS FETCH FIRST 3 ROWS ONLY
)

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길
9,3huwq6vwehb7,김하은,경상북도,포항시 북구,성곡서로 5번길
10,3hvd0w7gxl2a1o40,전이현,충청남도,아산시,영인로 5번길


- 두 개의 결과의 교집합을 구합니다.

In [17]:
%%SQL

SELECT * FROM member1
INTERSECT
(
    SELECT member_id, member_name, region, address1, address2 
    FROM member 
    WHERE member_id >= '3huv3i06r0kjq'
    ORDER BY member_id
    OFFSET 0 ROWS FETCH FIRST 3 ROWS ONLY
)

,member_id,member_name,region,address1,address2


- member1에서 MINUS 이후의 쿼리의 결과와의 차집합 연산을 수행합니다.

In [18]:
%%SQL

SELECT * FROM member1
MINUS
(
    SELECT member_id, member_name, region, address1, address2 
    FROM member 
    WHERE member_id >= '3huv3i06r0kjq'
    ORDER BY member_id
    OFFSET 0 ROWS FETCH FIRST 3 ROWS ONLY
)

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


# 윈도우 함수 


```SQL
SELECT 윈도우 함수(매개변수) OVER (연산 범위 구문)
FROM ...
```

**연산 범위 구문**

\[PARTITION BY 컬럼\] \[ORDER BY 절\] \[WINDOWING 절\]

|연산 범위 지정자|설명|
|-----|-----|
|PARTITION BY 컬럼|지정한 컬럼을 기준으로 현재 행의 값과 동일한 값을 지닌 행들이 범위|
|ORDER BY 절|범위로 지정된 행들의 정렬 순서|

윈도우 함수(Window Function)는 **각각의 행(Row)** 을 기준으로,

`OVER()` 절에서 지정한 **범위(Window)** 내의 데이터와 연산을 수행하여  

그 결과를 각 행에 반환하는 함수입니다.  

즉, "각 행마다 주변(같은 그룹 또는 지정 범위)의 데이터를 모아서 계산한 값"을 출력할 수 있습니다.  

이때, 집계 연산을 하더라도 행이 사라지지 않고 그대로 유지됩니다.

**윈도우함수**

<table>
  <tr>
    <th>종류</th><th>함수</th><th>설명</th>
  </tr>
  <tr>
    <td rowspan="3">순위</td><td>RANK()</td><td>순위</td>
  </tr>
  <tr>
    <td>DENSE_RANK()</td><td>순위, 동일한 순위를 하나의 건수로 취급</td>
  </tr>
  <tr>
    <td>ROW_NUMBER()</td><td>순서</td>
  </tr>
  <tr>
    <td rowspan="4">순서</td><td>FIRST_VALUE(표현)</td><td>첫번째 값</td>
  </tr>
  <tr>
    <td>LAST_VALUE(표현)</td><td>마지막 값</td>
  </tr>
  <tr>
    <td>LAG(표현[, 위치][, 기본값)</td><td>이전 값</td>
  </tr>
  <tr>
    <td>LEAD(표현[, 위치][, 기본값)</td><td>이후 값</td>
  </tr>
  <tr>
    <td rowspan="4">비율</td><td>RATIO_TO_REPORT(표현)</td><td>범위합에서 현재값의 비율</td>
  </tr>
  <tr>
    <td>PERCENT_RANK()</td><td>백분위</td>
  </tr>
  <tr>
    <td>CUME_DIST()</td><td>누적분위</td>
  </tr>
  <tr>
    <td>NTILE(N)</td><td>N분위수</td>
  </tr> 
</table>

**[예제 4]** 윈도우 함수의 기능을 알아봅니다.

In [19]:
%%SQL
SELECT * FROM book1 -- book1의 내용을 가져옵니다.

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- OVER에서 PARTITION BY와 같은 한정 구문을 넣지 않으면 모든 행을 window 함수의 대상 행으로 잡습니다.

In [20]:
%%SQL

SELECT book_title, publisher, pub_date, 
    rank() OVER (ORDER BY pub_date) as 출간순서
FROM book1

,book_title,publisher,pub_date,출간순서
1,상실의 시대,문학사상,2000-10-01,1
2,수상한 과학,풀빛,2004-01-01,2
3,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2004-07-01,3
4,"궁금해요, 세종 대왕",풀빛,2019-07-01,4
5,뉴스를 보는 눈,풀빛,2019-10-01,5
6,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2022-07-01,6
7,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2023-02-01,7
8,2024 SIMPLE 의료법규,군자출판사,2023-05-01,8
9,중학 국어 기초 완성,꿈을담는틀,2024-10-01,9
10,임신부·모유수유부의 안전한 약물 사용,군자출판사,2024-11-01,10


- OVER에서 PARTITION BY publisher를 통해 window 함수의 대상 행을 publisher가 같은 행으로 한정을 시킵니다.

In [21]:
%%SQL

SELECT book_title, publisher, pub_date, 
    rank() OVER (PARTITION BY publisher ORDER BY pub_date) as 출간순서
FROM book1
ORDER BY book_title

,book_title,publisher,pub_date,출간순서
1,2024 SIMPLE 의료법규,군자출판사,2023-05-01,2
2,"궁금해요, 세종 대왕",풀빛,2019-07-01,2
3,뉴스를 보는 눈,풀빛,2019-10-01,3
4,상실의 시대,문학사상,2000-10-01,1
5,수상한 과학,풀빛,2004-01-01,1
6,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2022-07-01,2
7,임신부·모유수유부의 안전한 약물 사용,군자출판사,2024-11-01,3
8,중학 국어 기초 완성,꿈을담는틀,2024-10-01,1
9,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2023-02-01,1
10,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2004-07-01,1


In [22]:
%%SQL

SELECT book_title, publisher, pub_date, extract(YEAR from pub_date) year FROM book
FETCH FIRST 12 ROWS ONLY -- 처음에 12개의 행만을 가져옵니다.

,book_title,publisher,pub_date,year
1,황소 아저씨,길벗어린이,2001-01-01,2001
2,명 1,갑을당,2001-02-01,2001
3,이승만 다시 보기,기파랑,2001-03-01,2001
4,뭐하니?,길벗어린이,2001-05-01,2001
5,돼지가 철학에 빠진 날,김영사,2001-07-01,2001
6,Object Oriented Perl,인포북,2001-08-01,2001
7,나는 이런 책을 읽어왔다,청어람미디어,2001-09-01,2001
8,내 친구 쟈쟈 4,케이시,2001-10-01,2001
9,〈그리고 두려운 것〉에 대한 인문학적 고찰,전남대학교출판문화원,2001-11-01,2001
10,에필로그,사이언스북스,2001-12-01,2001


In [23]:
%%SQL

SELECT * FROM (
    SELECT book_title, publisher, pub_date, extract(YEAR from pub_date) year FROM book
    FETCH FIRST 12 ROWS ONLY
) -- 가져온 12개을 인라인뷰 행태로 사용해본것입니다.

,book_title,publisher,pub_date,year
1,황소 아저씨,길벗어린이,2001-01-01,2001
2,명 1,갑을당,2001-02-01,2001
3,이승만 다시 보기,기파랑,2001-03-01,2001
4,뭐하니?,길벗어린이,2001-05-01,2001
5,돼지가 철학에 빠진 날,김영사,2001-07-01,2001
6,Object Oriented Perl,인포북,2001-08-01,2001
7,나는 이런 책을 읽어왔다,청어람미디어,2001-09-01,2001
8,내 친구 쟈쟈 4,케이시,2001-10-01,2001
9,〈그리고 두려운 것〉에 대한 인문학적 고찰,전남대학교출판문화원,2001-11-01,2001
10,에필로그,사이언스북스,2001-12-01,2001


- 위 인라인뷰로 rank 작업을 합니다. rank는 동일순위 일경우 다음 순위는 동일 순위 만큼 건너 뛴다음의 순위가 됩니다.

In [24]:
%%SQL

SELECT book_title, publisher, pub_date, 
    rank() OVER (PARTITION BY year ORDER BY pub_date) 연간순서 
FROM (
    SELECT book_title, publisher, pub_date, extract(YEAR from pub_date) year FROM book
    FETCH FIRST 12 ROWS ONLY
)

,book_title,publisher,pub_date,연간순서
1,황소 아저씨,길벗어린이,2001-01-01,1
2,명 1,갑을당,2001-02-01,2
3,이승만 다시 보기,기파랑,2001-03-01,3
4,뭐하니?,길벗어린이,2001-05-01,4
5,돼지가 철학에 빠진 날,김영사,2001-07-01,5
6,Object Oriented Perl,인포북,2001-08-01,6
7,나는 이런 책을 읽어왔다,청어람미디어,2001-09-01,7
8,내 친구 쟈쟈 4,케이시,2001-10-01,8
9,〈그리고 두려운 것〉에 대한 인문학적 고찰,전남대학교출판문화원,2001-11-01,9
10,에필로그,사이언스북스,2001-12-01,10


- 반면 dense_rank는 동일순위 일경우 다음 순위는 바로 그 다음 순위로 메겨집니다.

In [25]:
%%SQL

SELECT book_title, publisher, pub_date, 
    dense_rank() OVER (PARTITION BY year ORDER BY pub_date) 연간순위 
FROM (
    SELECT book_title, publisher, pub_date, extract(YEAR from pub_date) year FROM book
    FETCH FIRST 12 ROWS ONLY
)

,book_title,publisher,pub_date,연간순위
1,황소 아저씨,길벗어린이,2001-01-01,1
2,명 1,갑을당,2001-02-01,2
3,이승만 다시 보기,기파랑,2001-03-01,3
4,뭐하니?,길벗어린이,2001-05-01,4
5,돼지가 철학에 빠진 날,김영사,2001-07-01,5
6,Object Oriented Perl,인포북,2001-08-01,6
7,나는 이런 책을 읽어왔다,청어람미디어,2001-09-01,7
8,내 친구 쟈쟈 4,케이시,2001-10-01,8
9,〈그리고 두려운 것〉에 대한 인문학적 고찰,전남대학교출판문화원,2001-11-01,9
10,에필로그,사이언스북스,2001-12-01,10


- row_number는 1씩 무조건 증가하므로, 동일 여부와 상관없는 순위를 구하는 데 사용할 수 있습니다.

In [26]:
%%SQL

SELECT book_title, publisher, pub_date, 
    row_number() OVER (PARTITION BY year ORDER BY pub_date) 연간순위 
FROM (
    SELECT book_title, publisher, pub_date, extract(YEAR from pub_date) year FROM book
    FETCH FIRST 12 ROWS ONLY
)

,book_title,publisher,pub_date,연간순위
1,황소 아저씨,길벗어린이,2001-01-01,1
2,명 1,갑을당,2001-02-01,2
3,이승만 다시 보기,기파랑,2001-03-01,3
4,뭐하니?,길벗어린이,2001-05-01,4
5,돼지가 철학에 빠진 날,김영사,2001-07-01,5
6,Object Oriented Perl,인포북,2001-08-01,6
7,나는 이런 책을 읽어왔다,청어람미디어,2001-09-01,7
8,내 친구 쟈쟈 4,케이시,2001-10-01,8
9,〈그리고 두려운 것〉에 대한 인문학적 고찰,전남대학교출판문화원,2001-11-01,9
10,에필로그,사이언스북스,2001-12-01,10


In [27]:
%%SQL

SELECT book_title, publisher, pub_date, 
    CUME_DIST() OVER (PARTITION BY year ORDER BY pub_date) 연간순위 
FROM (
    SELECT book_title, publisher, pub_date, extract(YEAR from pub_date) year FROM book
    FETCH FIRST 12 ROWS ONLY
)

,book_title,publisher,pub_date,연간순위
1,황소 아저씨,길벗어린이,2001-01-01,0.1
2,명 1,갑을당,2001-02-01,0.2
3,이승만 다시 보기,기파랑,2001-03-01,0.3
4,뭐하니?,길벗어린이,2001-05-01,0.4
5,돼지가 철학에 빠진 날,김영사,2001-07-01,0.5
6,Object Oriented Perl,인포북,2001-08-01,0.6
7,나는 이런 책을 읽어왔다,청어람미디어,2001-09-01,0.7
8,내 친구 쟈쟈 4,케이시,2001-10-01,0.8
9,〈그리고 두려운 것〉에 대한 인문학적 고찰,전남대학교출판문화원,2001-11-01,0.9
10,에필로그,사이언스북스,2001-12-01,1


- book_id 별로 일련 번호(row_number)를 구하고 이 숫자가 5 미만을 출력하는 조건을 가해서 book_id 별로 4개가 넘지 않는 행을 출력합니다.

In [28]:
%%SQL
SELECT * FROM (
    SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
    FROM rating
) WHERE rnum < 5
FETCH FIRST 12 ROWS ONLY 

,book_id,member_id,rating,rate_date,rnum
1,1,1puljb4gk0,8,2024-07-26 11:05:20,1
2,1,1rx1bubjc7q4,10,2024-08-20 11:27:24,2
3,1,j5yufxvi7f9f,8,2024-09-11 11:15:46,3
4,1,lfj601xx7thzoq,10,2024-09-14 10:23:55,4
5,2,x87yayr2ofca,10,2019-10-10 16:24:41,1
6,2,s468i3x5s611l4m55,9,2019-10-14 13:50:53,2
7,2,nlnttauc0,9,2019-10-22 14:05:36,3
8,2,hj01motzmph7,8,2019-10-23 10:43:39,4
9,3,7ze9z3nb9rdcsnz,9,2020-09-02 10:29:06,1
10,3,ghvuyk3o34x,10,2020-10-16 15:12:33,2


- book_id 별로 일련 번호(row_number)를 구하고 이 숫자가 11 미만을 출력하는 조건을 가해서 book_id 별로 10개가 넘지 않는 행을 출력합니다.

In [29]:
%%SQL

SELECT * FROM (
    SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
    FROM rating
) WHERE rnum <= 10
FETCH FIRST 10 ROWS ONLY 

,book_id,member_id,rating,rate_date,rnum
1,1,1puljb4gk0,8,2024-07-26 11:05:20,1
2,1,1rx1bubjc7q4,10,2024-08-20 11:27:24,2
3,1,j5yufxvi7f9f,8,2024-09-11 11:15:46,3
4,1,lfj601xx7thzoq,10,2024-09-14 10:23:55,4
5,1,x70ttmt57m,9,2024-10-11 13:03:38,5
6,1,818y2x6ytu6,9,2024-10-31 13:01:20,6
7,1,mqe3hevxlprqvb,10,2024-10-31 15:16:49,7
8,1,7wk4zeha1we6,8,2024-11-03 15:49:09,8
9,1,ib436aek2p,9,2024-11-09 14:27:54,9
10,1,ds69rkt6qdwahf,9,2025-02-02 13:52:11,10


- 윈도우 함수의 다른 응용입니다. 일반 집계함수에서도 윈도우 함수로 활용있는 것이 있고, LAG, LEAD 는 파라메터를 전달하여 사용할 수 있습니다. 
    
  LAG 정렬 기준으로 지정한 수만큼의 이전 수를 가지고오, LEAD에서 지정한 수만큼의 다음 수를 가져 옵니다.


In [30]:
%%SQL
SELECT * FROM (
    SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
    FROM rating
) WHERE rnum <= 10
FETCH FIRST 12 ROWS ONLY 

,book_id,member_id,rating,rate_date,rnum
1,1,1puljb4gk0,8,2024-07-26 11:05:20,1
2,1,1rx1bubjc7q4,10,2024-08-20 11:27:24,2
3,1,j5yufxvi7f9f,8,2024-09-11 11:15:46,3
4,1,lfj601xx7thzoq,10,2024-09-14 10:23:55,4
5,1,x70ttmt57m,9,2024-10-11 13:03:38,5
6,1,818y2x6ytu6,9,2024-10-31 13:01:20,6
7,1,mqe3hevxlprqvb,10,2024-10-31 15:16:49,7
8,1,7wk4zeha1we6,8,2024-11-03 15:49:09,8
9,1,ib436aek2p,9,2024-11-09 14:27:54,9
10,1,ds69rkt6qdwahf,9,2025-02-02 13:52:11,10


In [31]:
%%SQL

SELECT book_id, member_id, rating, 
    MAX(rating) OVER (PARTITION BY book_id) "도서별 최고평점", 
    AVG(rating) OVER (PARTITION BY book_id) "도서별 평균평점",
    LAG(rating, 1) OVER (PARTITION BY book_id ORDER BY rate_date) "이전 도서평점",
    LEAD(rating, 1) OVER (PARTITION BY book_id ORDER BY rate_date) "이후 도서평점"
FROM (
    SELECT * FROM (
        SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
        FROM rating
    ) WHERE rnum <= 10
    FETCH FIRST 12 ROWS ONLY 
)

,book_id,member_id,rating,도서별 최고평점,도서별 평균평점,이전 도서평점,이후 도서평점
1,1,1puljb4gk0,8,10,9,NULL,10.0
2,1,1rx1bubjc7q4,10,10,9,8.0,8.0
3,1,j5yufxvi7f9f,8,10,9,10.0,10.0
4,1,lfj601xx7thzoq,10,10,9,8.0,9.0
5,1,x70ttmt57m,9,10,9,10.0,9.0
6,1,818y2x6ytu6,9,10,9,9.0,10.0
7,1,mqe3hevxlprqvb,10,10,9,9.0,8.0
8,1,7wk4zeha1we6,8,10,9,10.0,9.0
9,1,ib436aek2p,9,10,9,8.0,9.0
10,1,ds69rkt6qdwahf,9,10,9,9.0,NULL


**WINDOWING 절**

- 지정된 행들의 범위를 현재 행의 위치를 기준으로 한정 (※ SQL Server 미지원)|
- \[BETWEEN 사용\]
> ROWS | RANGE BETWEEN UNBOUNDED PRECEDING | CURRENT ROW | VALUE_EXPR PRECEDING/ FOLLOWING
> 
>       AND UNBOUNDED FOLLOWING | CURRENT ROW | VALUE_EXPR PRECEDING/ FOLLOWING

- \[BETWEEN 미사용\] : 지정한 행부터 현재 행까지
> ROWS | RANGE UNBOUNDED PRECEDING | CURRENT ROW | VALUE_EXPR PRECEDING


**[예제 5]** WINDOWING 기능을 이용하여 각 행에서 정렬 순선를 기준으로 여러개의 행을 대상으로 지정하고 집계 연산을 수행합니다.

- BETWEEN ROWS를 사용하면 범위를 행단위 지정할 수 있 있습니다. 현재행을 기준으로 2 PRECEDING는 2행 선행하부터 CURRENT ROW 현재행 까지를 지정합니다.

In [32]:
%%SQL

SELECT * FROM (
    SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
    FROM rating
) WHERE rnum <= 10
FETCH FIRST 10 ROWS ONLY 

,book_id,member_id,rating,rate_date,rnum
1,1,1puljb4gk0,8,2024-07-26 11:05:20,1
2,1,1rx1bubjc7q4,10,2024-08-20 11:27:24,2
3,1,j5yufxvi7f9f,8,2024-09-11 11:15:46,3
4,1,lfj601xx7thzoq,10,2024-09-14 10:23:55,4
5,1,x70ttmt57m,9,2024-10-11 13:03:38,5
6,1,818y2x6ytu6,9,2024-10-31 13:01:20,6
7,1,mqe3hevxlprqvb,10,2024-10-31 15:16:49,7
8,1,7wk4zeha1we6,8,2024-11-03 15:49:09,8
9,1,ib436aek2p,9,2024-11-09 14:27:54,9
10,1,ds69rkt6qdwahf,9,2025-02-02 13:52:11,10


In [33]:
%%SQL

SELECT book_id, member_id, rating, rate_date,
    (
        NVL(rating, 0) + 
        NVL(LAG(rating, 1) OVER (PARTITION BY book_id ORDER BY rate_date), 0) +
        NVL(LAG(rating, 2) OVER (PARTITION BY book_id ORDER BY rate_date), 0)
    ) / (
        1 + 
        CASE WHEN LAG(rating, 1) OVER (PARTITION BY book_id ORDER BY rate_date) IS NULL THEN 0 ELSE 1 END +
        CASE WHEN LAG(rating, 2) OVER (PARTITION BY book_id ORDER BY rate_date) IS NULL THEN 0 ELSE 1 END
    ) "3일간 평균평점(직접)",
    AVG(rating) OVER (PARTITION BY book_id ORDER BY rate_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) "3일간 평균평점"
FROM (
    SELECT * FROM (
        SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
        FROM rating
    ) WHERE rnum <= 10
    FETCH FIRST 10 ROWS ONLY 
)

,book_id,member_id,rating,rate_date,3일간 평균평점(직접),3일간 평균평점
1,1,1puljb4gk0,8,2024-07-26 11:05:20,8,8
2,1,1rx1bubjc7q4,10,2024-08-20 11:27:24,9,9
3,1,j5yufxvi7f9f,8,2024-09-11 11:15:46,8.66666666666666666666666666666666666667,8.66666666666666666666666666666666666667
4,1,lfj601xx7thzoq,10,2024-09-14 10:23:55,9.33333333333333333333333333333333333333,9.33333333333333333333333333333333333333
5,1,x70ttmt57m,9,2024-10-11 13:03:38,9,9
6,1,818y2x6ytu6,9,2024-10-31 13:01:20,9.33333333333333333333333333333333333333,9.33333333333333333333333333333333333333
7,1,mqe3hevxlprqvb,10,2024-10-31 15:16:49,9.33333333333333333333333333333333333333,9.33333333333333333333333333333333333333
8,1,7wk4zeha1we6,8,2024-11-03 15:49:09,9,9
9,1,ib436aek2p,9,2024-11-09 14:27:54,9,9
10,1,ds69rkt6qdwahf,9,2025-02-02 13:52:11,8.66666666666666666666666666666666666667,8.66666666666666666666666666666666666667


- RANGE를 사용하면 ORDER BY의 값을 기준으로 범위를 정할 수 있습니다. 여기서 rnum / 2 이므로 RANGE BETWEEN 1 PRECEDING을 값을 기준으로 1을 앞선 행들을 WINDOW 함수의 대상이 됩니다.

In [34]:
%%SQL

SELECT book_id, member_id, rating, rnum / 2,
    AVG(rating) OVER (PARTITION BY book_id ORDER BY rnum / 2 RANGE BETWEEN 1 PRECEDING AND CURRENT ROW) "평균평점"
FROM (
    SELECT * FROM (
        SELECT book_id, member_id, rating, rate_date, row_number() OVER (PARTITION BY book_id ORDER BY rate_date) rnum
        FROM rating
    ) WHERE rnum <= 10
    FETCH FIRST 10 ROWS ONLY 
)
ORDER BY rate_date

,book_id,member_id,rating,RNUM/2,평균평점
1,1,1puljb4gk0,8,0.5,8
2,1,1rx1bubjc7q4,10,1,9
3,1,j5yufxvi7f9f,8,1.5,8.66666666666666666666666666666666666667
4,1,lfj601xx7thzoq,10,2,9.33333333333333333333333333333333333333
5,1,x70ttmt57m,9,2.5,9
6,1,818y2x6ytu6,9,3,9.33333333333333333333333333333333333333
7,1,mqe3hevxlprqvb,10,3.5,9.33333333333333333333333333333333333333
8,1,7wk4zeha1we6,8,4,9
9,1,ib436aek2p,9,4.5,9
10,1,ds69rkt6qdwahf,9,5,8.66666666666666666666666666666666666667


- 정렬 기준이 일자일 경우 일단위 지정이 가능합니다.

In [35]:
%%SQL

SELECT book_id, member_id, rate_date,
    AVG(rating) OVER (ORDER BY rate_date RANGE BETWEEN INTERVAL '7' DAY PRECEDING AND CURRENT ROW) "7일간 평균평점"
FROM rating
ORDER BY rate_date

,book_id,member_id,rate_date,7일간 평균평점
1,3558,8i12a0eiw42,2019-10-02 16:22:50,10
2,1680,qhj4eru9auejwz,2019-10-04 09:30:25,10
3,3328,qjms3d4nqba8,2019-10-04 10:02:33,9.33333333333333333333333333333333333333
4,1928,xjuw2w3kqv2,2019-10-04 13:07:17,9.5
5,1940,urob00uebq0,2019-10-04 16:04:36,9.4
...,...,...,...,...
103400,1080,hbr7tjgd9r,2025-02-20 16:47:31,8.61059907834101382488479262672811059908
103401,1905,ec19l8tnw5,2025-02-20 16:49:31,8.61149425287356321839080459770114942529
103402,4,a2ymxxu1a,2025-02-20 16:57:49,8.61467889908256880733944954128440366972
103403,2474,du99v8ap0rc2,2025-02-20 16:58:28,8.6155606407322654462242562929061784897


- BETWEEN이 아니면 지정 만큼 이전 부터의 데이터에서 현재행까지가 WINDOW 함수의 연산 대상이 됩니다.

In [36]:
%%SQL

SELECT book_id, member_id, rate_date,
    AVG(rating) OVER (ORDER BY rate_date RANGE INTERVAL '7' DAY PRECEDING) "7일간 평균평점"
FROM rating
ORDER BY rate_date

,book_id,member_id,rate_date,7일간 평균평점
1,3558,8i12a0eiw42,2019-10-02 16:22:50,10
2,1680,qhj4eru9auejwz,2019-10-04 09:30:25,10
3,3328,qjms3d4nqba8,2019-10-04 10:02:33,9.33333333333333333333333333333333333333
4,1928,xjuw2w3kqv2,2019-10-04 13:07:17,9.5
5,1940,urob00uebq0,2019-10-04 16:04:36,9.4
...,...,...,...,...
103400,1080,hbr7tjgd9r,2025-02-20 16:47:31,8.61059907834101382488479262672811059908
103401,1905,ec19l8tnw5,2025-02-20 16:49:31,8.61149425287356321839080459770114942529
103402,4,a2ymxxu1a,2025-02-20 16:57:49,8.61467889908256880733944954128440366972
103403,2474,du99v8ap0rc2,2025-02-20 16:58:28,8.6155606407322654462242562929061784897


# TOP N 쿼리
|방법|설명|
|----|----|
|ROWNUM<br/>슈도(pseudo) 컬럼|Oracle DBMS에서 사용하는 각각에 행에 부여된 일련 번호<br/>ROWNUM과 WHERE를 이용하여 행의 위치 기반으로 선택<br/>ORDER BY 과정이전에 부여<br/>ORDER BY를 사용하지 않았다면 행의 순서와 ROWNUM은 일치<br/>ORDER BY를 사용한다면 불일치 → INLINE VIEW로 만든 후에 선택|
|TOP (행의 수) \[PERCENT\] \[WITH TIES\]|SQL Server에서 사용<br/>PERCENT: 퍼센트 단위로 선택<br/>WITH TIES: ORDER BY 사용시 사용, 마지막 행과 이후 행과의 정렬 기준값이 같을 경우 포함|
|ROW LIMITING 절|ORACLE 12.1 버전에서 지원<br/>\[OFFSET offset \{ROW \| ROWS\}\]<br/>\[FETCH \{FIRST \| NEXT\} \[\{rowcount \| percent PERCENT\}\] {ROW \| ROWS} \{ONLY \| WITH TIES\}<br/>OFFSET: 건너뛸 행의 수, FETCH: 불러올 행의 수 지정|


**[예제 6]** 불러올 행의 위치와 수를 제어하는 구문에 대해서 알아봅니다.

- Oracle에는 행의 번호를 나타내 주는 ROWNUM이란 기본적으로 내장된 컬럼(pseudo 컬럼)이 있는데요,

  Oracle이 12.1 이전에서는 이 ROWNUM를 이용하여 TOP N과 같은 기능을 구현했습니다.

- ROWNUM 컬럼을 사용해봅니다.

In [37]:
%%SQL

SELECT rownum, book1.* 
FROM book1

,rownum,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- SQL에서는 열 선택 과정은 정렬(ORDER BY)보다 앞선 단계에서 이루어지기 때문에 ORDER BY를 하면 ROWNUM는 순서에 맞춰진 상태가 아닙니다.

In [38]:
%%SQL

SELECT rownum, book1.*
FROM book1
ORDER BY pub_date

,rownum,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
2,1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
3,2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
4,4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54
10,9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39


- 이 상태에서는 정렬에 순서에 따른 행을 선택할 수 없습니다.

In [39]:
%%SQL

SELECT rownum, book1.*
FROM book1
WHERE rownum < 5
ORDER BY pub_date

,rownum,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
2,1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
3,2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
4,4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31


- Oracle에서는 대상 테이블 또는 inline view의 행을 가져오면서, WHERE에서의 선택조건에 해당하는 행들을 추려내는 작업을 수행합니다.

  rownum은 WHERE에서 선택된 행의 개수에 따라 누적됩니다.

In [40]:
%%SQL

SELECT rownum, a.* FROM (
    SELECT book1.*
    FROM book1
    ORDER BY pub_date
) a
WHERE rownum < 5

,rownum,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,1,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
2,2,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
3,3,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
4,4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31


- Oracle의 rownum 은 1부터 시작하는데 WHERE에서 rownum이 2이상을 가져오도록 하고 있어, rownum은 버려집니다. 계속 rownum은 1이 되기 때문에,

  비어있는 결과를 얻게 되는 것입니다.

In [41]:
%%SQL

SELECT rownum, a.* FROM (
    SELECT book1.*
    FROM book1
    ORDER BY pub_date
) a
WHERE rownum >= 2

,rownum,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date


- 이런 동작에 의해서,  서브 쿼리 내에서 rownum을 만들어 놓고 상위 쿼리에서 처음 위치의 행을 수를 지정할 수 있습니다.

In [42]:
%%SQL

SELECT * FROM (
    SELECT rownum rnum, a.* FROM (
        SELECT book1.*
        FROM book1
        ORDER BY pub_date
    ) a
    WHERE rownum < 5
)
WHERE rnum >= 2

,rnum,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,3,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31


- Oracle 12.1 버젼에서는 불편한 rownum을 사용하지 않아도 행의 위치 기준으로 가져올 수 있는 기능을 제공합니다.

OFFSET \[시작 위치\] ROWS FETCH NEXT \[가져올 데이터 수\] ROWS ONLY

In [43]:
%%SQL

SELECT book1.*
FROM Book1
ORDER BY pub_date
OFFSET 1 ROWS FETCH NEXT 4 ROWS ONLY

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
4,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43


# PIVOT 절

행단위의 내용을 열단위(컬럼 레벨)로 정리합니다.

```SQL
PIVOT [XML](
	집계함수 (표현) [[AS] alias] [, 집계함수2(표현2) [[AS] alias2] …
	FOR {컬럼|(컬럼1[, 컬럼2]…)}
	IN ({
		{표현| (표현1, 표현2…)} [[AS] alias]} … 
		| 서브쿼리
		| ANY[, ANY]…
	})
)
```
집계 함수는 행(FOR에 설정되지 않은 항목)과 열(FOR 에서 설정된 항목)에 의해 교차에 의해 생긴 셀에 대응 되는 값들을 처리해 하나의 대표값으로 만들어 주는 함수

FOR 에서는 열단위 배치의 기준이 되는 컬럼

IN 에서는 FOR에서 열단위로 정리한 컬럼값을 지정 합니다.

**서브쿼리와, ANY는 XML구를 넣을 경우 사용 가능**

**[예제 7]** PIVOT의 여러 용법을 알아봅니다.

In [44]:
%%SQL

SELECT * FROM book1

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- IN의 표현식은 기본적으로 고정 개수이기 때문에 PIVOT에서 다루지 않을 케이스는 기타가 되도록 설정했습니다.

In [45]:
%%SQL

SELECT * 
FROM (
    SELECT EXTRACT(YEAR FROM pub_date) year, 
        CASE 
            WHEN publisher NOT IN ('군자출판사', '풀빛', '책읽는고양이') THEN '기타'
            ELSE publisher 
        END publisher
    FROM book1
)
PIVOT (
    COUNT(*) 빈도 FOR publisher IN ('군자출판사', '풀빛', '책읽는고양이')
)

,year,'군자출판사'_빈도,'풀빛'_빈도,'책읽는고양이'_빈도
1,2004,0,1,1
2,2000,0,0,0
3,2019,0,2,0
4,2022,0,0,1
5,2023,2,0,0
6,2024,1,0,0


- PIVOT의 수행은 컬럼의 선택 과정 보다 먼저 이루어집니다. 그리고 복수의 집계작업을 할 수 있습니다.

In [46]:
%%SQL

SELECT book_id, nvl("2024_평가수", 0), round("2024_평균평점", 2), nvl("2025_평가수", 0), round("2025_평균평점", 2) FROM (
    SELECT book_id, EXTRACT(YEAR FROM rate_date) 연도, rating 
    FROM rating
) PIVOT (
    COUNT(*) 평가수, AVG(rating) 평균평점 FOR 연도 IN (2024, 2025)
)

,book_id,"NVL(""2024_평가수"",0)","ROUND(""2024_평균평점"",2)","NVL(""2025_평가수"",0)","ROUND(""2025_평균평점"",2)"
1,6,6,8.67,1,7
2,14,15,8.53,0,NULL
3,23,53,8.66,5,8.6
4,27,8,9.13,2,9.5
5,50,2,9.5,1,10
...,...,...,...,...,...
1681,3280,10,8.7,1,8
1682,2808,0,NULL,0,NULL
1683,2819,1,8,0,NULL
1684,3021,0,NULL,0,NULL


- 여러 개의 기준을 사용하여 컬럼을 구성할 수 있습니다.

In [47]:
%%SQL

SELECT * FROM (
    SELECT book_id, EXTRACT(YEAR FROM rate_date) year, 
        CASE 
            WHEN EXTRACT(MONTH FROM rate_date) <= 6 THEN '상반기'
            ELSE '하반기'
        END as 반기구분, rating
    FROM rating
)
PIVOT (
    AVG(Rating) 평균평점 FOR (year, 반기구분) IN ((2023, '상반기'), (2023, '하반기'), (2024, '상반기'), (2024, '하반기'))
)

,book_id,2023_'상반기'_평균평점,2023_'하반기'_평균평점,2024_'상반기'_평균평점,2024_'하반기'_평균평점
1,6,8.625,8.8,8.75,8.5
2,14,8.25,7.83333333333333333333333333333333333333,9,8.36363636363636363636363636363636363636
3,23,8.94285714285714285714285714285714285714,8.80769230769230769230769230769230769231,8.68,8.64285714285714285714285714285714285714
4,27,NULL,NULL,NULL,9.125
5,50,NULL,9.5,10,9
...,...,...,...,...,...
1681,3280,8.66666666666666666666666666666666666667,9.5,8.8,8.6
1682,2808,NULL,NULL,NULL,NULL
1683,2819,9.66666666666666666666666666666666666667,8.75,NULL,8
1684,3021,NULL,NULL,NULL,NULL


- ORACLE에서는 컬럼의 구성이 가변적인 쿼리는 성립하지 않습니다. 

따라서 PIVOT에서 IN에 쿼리나 임의의 요소를 대상으로 하려면 XML으로 설정하면, _xml 컬럼 단일컬럼으로 이하 컬럼내용을 하나의 xml로 만들어서 전달합니다.

In [48]:
%%SQL

SELECT * 
FROM (SELECT EXTRACT(YEAR FROM pub_date), publisher FROM book1)
PIVOT XML (
    COUNT(*) 빈도 FOR publisher IN (ANY)
)

,EXTRACT(YEARFROMPUB_DATE),publisher_xml
1,2000,"<PivotSet><item><column name = ""PUBLISHER"">문학사..."
2,2004,"<PivotSet><item><column name = ""PUBLISHER"">책읽는..."
3,2019,"<PivotSet><item><column name = ""PUBLISHER"">풀빛<..."
4,2022,"<PivotSet><item><column name = ""PUBLISHER"">책읽는..."
5,2023,"<PivotSet><item><column name = ""PUBLISHER"">군자출..."
6,2024,"<PivotSet><item><column name = ""PUBLISHER"">군자출..."


# UNPIVOT 절

컬럼 단위의 내용을 열단위 내용으로 전환시켜주는 연산입니다.

```SQL
UNPIVOT [{INCLUDE | EXCLUDE} NULLS] (
	{값컬럼| (값컬럼1[, 값컬럼2]…)}
	FOR {컬럼| (컬럼1, [, 컬럼2] …)}
	IN (
	    {전환컬럼|(전환컬럼1[, 전환컬럼2]…)}[AS {리터럴|(리터컬1[, 리터럴2]…)}]
	)
)
```
값컬럼은 컬럼에 담겨 있는 값이 들어갈 컬럼명입니다.

FOR는 컬럼을 나타내는 값이 들어갈 컬럼명입니다.

IN 에서는 행단위로 전환시킬 컬럼을 정의 합니다.

연산에서 지정이 되는 않은 컬럼의 내용은 해당 내용이 UNPIVOT된 내용에 반복됩니다.

In [49]:
%%SQL

SELECT * FROM bookstat

,book_id,stat_ymd,rent_count,rating_sum,rating_count,stat_date
1,142,20220929,1,24,3,2025-09-12 04:18:07
2,3405,20240308,1,29,3,2025-09-12 04:18:07
3,2222,20220408,1,28,3,2025-09-12 04:18:07
4,2046,20250126,1,27,3,2025-09-12 04:18:07
5,2184,20221029,1,25,3,2025-09-12 04:18:07
6,1895,20230703,1,25,3,2025-09-12 04:18:07
7,1607,20210723,1,27,3,2025-09-12 04:18:07
8,1111,20230625,1,26,3,2025-09-12 04:18:07
9,270,20230213,1,28,3,2025-09-12 04:18:07


**[예제 8]**  UNPIVOT의 기능 파악해봅니다.

In [50]:
%%SQL

SELECT book_id, substr(stat_ymd, 1, 4) as year, rent_count, rating_count 
FROM bookstat -- bookstat의 내용을 봅니다.

,book_id,year,rent_count,rating_count
1,142,2022,1,3
2,3405,2024,1,3
3,2222,2022,1,3
4,2046,2025,1,3
5,2184,2022,1,3
6,1895,2023,1,3
7,1607,2021,1,3
8,1111,2023,1,3
9,270,2023,1,3


- bookstat의 rent_count와 rating_count를 행단위로 전환시킵니다. 

    컬럼 구분값이 들어가는 컬럼을 '구분', 그리고 컬럼값이 들어가는 컬럼을 '건수' 컬럼으로 지정합니다.

    rent_count는 'RENT'로 rating_count는 'RATING'으로 컬럼 구분값으로 만들어 UNPIVOT을 합니다.

In [51]:
%%SQL

SELECT *
FROM (
    SELECT book_id, substr(stat_ymd, 1, 4) as year, rent_count, rating_count 
    FROM bookstat
)
UNPIVOT (
    건수 FOR 구분 IN (rent_count AS 'RENT', rating_count AS 'RATING')
)

,book_id,year,구분,건수
1,142,2022,RENT,1
2,142,2022,RATING,3
3,3405,2024,RENT,1
4,3405,2024,RATING,3
5,2222,2022,RENT,1
6,2222,2022,RATING,3
7,2046,2025,RENT,1
8,2046,2025,RATING,3
9,2184,2022,RENT,1
10,2184,2022,RATING,3


- 컬럼그룹을 만들어 UNPIVOT을 수행합니다.

In [52]:
%%SQL

SELECT * FROM (
    SELECT book_id, EXTRACT(YEAR FROM rate_date) year, 
        CASE 
            WHEN EXTRACT(MONTH FROM rate_date) <= 6 THEN '상반기'
            ELSE '하반기'
        END as 반기구분, rating
    FROM rating
)
PIVOT (
    AVG(Rating) 평균평점 FOR (year, 반기구분) IN ((2023, '상반기'), (2023, '하반기'), (2024, '상반기'), (2024, '하반기'))
)

,book_id,2023_'상반기'_평균평점,2023_'하반기'_평균평점,2024_'상반기'_평균평점,2024_'하반기'_평균평점
1,6,8.625,8.8,8.75,8.5
2,14,8.25,7.83333333333333333333333333333333333333,9,8.36363636363636363636363636363636363636
3,23,8.94285714285714285714285714285714285714,8.80769230769230769230769230769230769231,8.68,8.64285714285714285714285714285714285714
4,27,NULL,NULL,NULL,9.125
5,50,NULL,9.5,10,9
...,...,...,...,...,...
1681,3280,8.66666666666666666666666666666666666667,9.5,8.8,8.6
1682,2808,NULL,NULL,NULL,NULL
1683,2819,9.66666666666666666666666666666666666667,8.75,NULL,8
1684,3021,NULL,NULL,NULL,NULL


In [53]:
%%SQL

SELECT * FROM (
    SELECT * FROM (
        SELECT book_id, EXTRACT(YEAR FROM rate_date) year, 
            CASE 
                WHEN EXTRACT(MONTH FROM rate_date) <= 6 THEN '상반기'
                ELSE '하반기'
            END as 반기구분, rating
        FROM rating
    )
    PIVOT (
        AVG(Rating) 평균평점 FOR (year, 반기구분) IN ((2023, '상반기'), (2023, '하반기'), (2024, '상반기'), (2024, '하반기'))
    )
)
UNPIVOT (
    (상반기, 하반기) FOR 연도 IN (("2023_'상반기'_평균평점", "2023_'하반기'_평균평점") AS 2023, ("2024_'상반기'_평균평점", "2024_'하반기'_평균평점") AS 2024)
)

,book_id,연도,상반기,하반기
1,6,2023,8.625,8.8
2,6,2024,8.75,8.5
3,14,2023,8.25,7.83333333333333333333333333333333333333
4,14,2024,9,8.36363636363636363636363636363636363636
5,23,2023,8.94285714285714285714285714285714285714,8.80769230769230769230769230769230769231
...,...,...,...,...
2671,3280,2024,8.8,8.6
2672,2819,2023,9.66666666666666666666666666666666666667,8.75
2673,2819,2024,NULL,8
2674,3026,2023,8.3,8.7


# 정규 표현식

정규 표현식(Regular Expression, RegEx)은
문자열에서 **특정한 규칙(패턴)**을 정의한 식입니다.

정규 표현식을 이용하면 검색·추출·치환을 효과적으로 수행할 수 있습니다.

**문자정의식**

**\[\[^\]{문자|문자1-문자2|문자셋}\[,{문자|문자1-문자2|문자셋}\]\]**


**정규표현식 함수**

|함수|설명|
|---|----|
|REGEXP_LIKE(source_char, pattern\[,match_param\])|정규표현식에 해당 여부|
|REGEXP_REPLACE(source_char, pattern\[, replace_string\[, position\[, occurrence\[,match_param\]\]\]\])|정규표현식에 해당 문자열을 치환하여 반환<br/>replace_string: 치환 패턴, \n n번째 서브패턴|
|REGEXP_SUBSTR(source_char, pattern\[, position\[, occurrence\[, match_param\[, subexpr\]\]\]\])|정규표현식에 해당 패턴을 반환 subexpr(0: 전체, 1이상: 서브패턴 순서)|
|REGEXP_INSTR(source_char, pattern\[, position\[, occurrence\[, match_param\[, subexpr\]\]\]\])|정규표현식에 해당 패턴의 시작 위치 반환|
|REGEXP_COUNT(source_char, pattern\[, position\[, match_param\]\])|정규표현식의 패턴이 등장한 횟수|

**source_char**: 대상 컬럼

**pattern**: 정규 표현

**position**: 탐색 시작 위치

**occurrence**: 일치 횟수

**match_param**(기본값 c)

> i – 대소문자 비구분, c –대소문자 구분, n – dot(.)에 개행문자 포함, m-다중행 모드,
>
> x- 검색 패턴에서 공백문자 무시
> 
> 여러 옵션 문자를 조합해서 구성 가능


**[예제 9]** 정규 표현식 함수의 기능을 알아봅니다.

- REGEXP_LIKE 정규 표현식의 패턴에 해당하는 행들을 가져옵니다.

In [54]:
%%SQL

SELECT contents FROM bookdetail

,contents
1,“오늘은 한그릇으로 간단하게 해결하자. \n그래도 탄.단.지 밸런스는 맞춰 건강하게...
2,"컬러 채소와 10가지 잡곡으로 짓는 55가지 맛있는 채소 솥밥\n■ 화이트, 옐로,..."
3,베스트셀러 1위! 2025 뿌미맘 가계부 출시\n출시 기념 30% 즉시 할인!\n2...
4,미국 아마존 금융ㆍ사업 분야 1위 \n국내 유명서점 10년간 종합 베스트셀러\n가장...
5,"선진국에서는 학교, 자선단체 등 비영리단체의 경영혁신이 선풍을 일으키고 있다. 이 ..."
...,...
71,왜 일해야 하는가?\n어째서 이토록 우리네 삶이 고달픈가?\n무슨 뾰족한 수는 없는...
72,"“수림아, 어떤 사람이 어른인지 아니?”\n자기 힘으로 살아 보려고 애쓰는 사람이야..."
73,29만 명의 구독자와 2만 명의 수강생이 증명한 최고의 C 언어 강의\n나도코딩의 ...
74,최신 프런트엔드 웹 디자인은 물론 인터랙티브 웹 사이트 8개를 한 번에 완성할 수 ...


In [55]:
%%SQL

SELECT contents FROM bookdetail WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 5 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필독서다...."
2,“최고의 독서교육법은 실행 가능한 독서법입니다”\n\n창비 좋은 어린이책 대상을 수...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."
4,"4·19혁명, 유신체제, 광주항쟁, 6월항쟁 등\n현대사의 주요 쟁점에 관한 서술 ..."
5,"뮤직 엔터테이너 송사비가 들려주는, 쉽고 재미있는 클래식 음악 야화\n\n음악의 아..."


- 정규 표현식이 나타낸 부분을 치환합니다. 치환한 할 문자의 기본은 널 문자입니다.

In [56]:
%%SQL

SELECT REGEXP_REPLACE(contents, '독서') contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필다.”\..."
2,“최고의 교육법은 실행 가능한 법입니다”\n\n창비 좋은 어린이책 대상을 수상한 교...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- 바로 다음에 치환할 문자를 지정할 수 있습니다.

In [57]:
%%SQL

SELECT REGEXP_REPLACE(contents, '독서', 'Reading') contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필Read..."
2,“최고의 Reading교육법은 실행 가능한 Reading법입니다”\n\n창비 좋은 ...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- 위치화 치환 횟수를 지정할 수 있습니다

In [58]:
%%SQL

SELECT REGEXP_REPLACE(contents, '독서', 'Reading', 1, 1) contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필Read..."
2,“최고의 Reading교육법은 실행 가능한 독서법입니다”\n\n창비 좋은 어린이책 ...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- 두 번째에서 1회만 치환하게 합니다.

In [59]:
%%SQL

SELECT REGEXP_REPLACE(contents, '독서', 'Reading', 1, 2) contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필독서다...."
2,“최고의 독서교육법은 실행 가능한 Reading법입니다”\n\n창비 좋은 어린이책 ...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- REGEXP_SUBSTR은 정규 표현식이 나타내는 문자열만을 가져옵니다.

  정규 표현식에서 .은 특수한 역할을 합니다. .은 개행 문자(\n, newline)이 아닌 모든 문자를 나타냅니다. 그래서 개행 문자를 만나기 전까지의 문자열을 나타냅니다.



In [60]:
%%SQL

SELECT REGEXP_SUBSTR(contents, '.+', 1, 1) contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필독서다.”"
2,“최고의 독서교육법은 실행 가능한 독서법입니다”
3,"전 세계 3,000만 독자에게"


In [61]:
%%SQL

SELECT REGEXP_SUBSTR(contents, '.+', 1, 2) contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“영미권에서 60년간 널린 읽힌 공부법의 고전서에서도, 세계적인 베스트셀러이자 지금..."
2,창비 좋은 어린이책 대상을 수상한 독서교육 전문가 최승필 작가가
3,지울 수 없는 후유증을 남긴 신드롬의 시작!


- .이 개행 문자 또한 일반 문자처럼 취급이 될 수 있게 하는 옵션이 'n' 입니다.

In [62]:
%%SQL

SELECT REGEXP_SUBSTR(contents, '.+', 1, 1, 'n') contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필독서다...."
2,“최고의 독서교육법은 실행 가능한 독서법입니다”\n\n창비 좋은 어린이책 대상을 수...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- .이 개행 문자 또한 일반 문자처럼 취급이 될 수 있게 하는 옵션이 'n' 입니다. 두 번째 문자열은 없습니다.

In [63]:
%%SQL

SELECT REGEXP_SUBSTR(contents, '.+', 1, 2, 'n') contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,NULL
2,NULL
3,NULL


- REGEXP_INSTR은 패턴의 위치를 반환합니다.

In [64]:
%%SQL

SELECT SUBSTR(contents, REGEXP_INSTR(contents, '독서', 1, 2)) contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,contents
1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필독서다...."
2,독서법입니다”\n\n창비 좋은 어린이책 대상을 수상한 독서교육 전문가 최승필 작가가...
3,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- REGEXP_COUNT는 패턴이 등장한 횟수를 반환합니다.

In [65]:
%%SQL

SELECT REGEXP_COUNT(contents, '독서'), contents contents 
FROM bookdetail 
WHERE REGEXP_LIKE(contents, '독서') FETCH FIRST 3 ROWS ONLY

,"REGEXP_COUNT(CONTENTS,'독서')",contents
1,1,"“이 책은 중학생부터 100세 노인까지, 평생 곁에 두고 봐야 하는 영어 필독서다...."
2,22,“최고의 독서교육법은 실행 가능한 독서법입니다”\n\n창비 좋은 어린이책 대상을 수...
3,1,"전 세계 3,000만 독자에게\n지울 수 없는 후유증을 남긴 신드롬의 시작!\n『미..."


- book_title에서 '패턴: 국어' 이 나오는 행만 가져옵니다.

In [66]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '국어')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,413,신공략 중국어 독해 초급편,북경어언대학 출판사편,210,다락원,김종호|강희명,대외 중국어교육 전문기관 북경어언대학의 단계별 독해 교재 중 첫 번째 단계이다. 각...,2005-09-01,2005-09-01 12:37:36
2,1602,파아란 중국어교실 1단계 C,어린이를 위한 정통 중국어 첫걸음 길라잡이,765,상록수,헨드리크 빌렘 반 룬,는 중국 국내 초ㆍ중등생들의 어문교재를 기준으로 총 3단계로 나뉘어져 있고 각 단계...,2006-04-01,2006-04-01 11:25:50
3,578,국어과 교수 학습 방법,NULL,282,역락,최지현|서혁|심영택|이도영|최미숙|김정자|김혜정,『국어과 교수 학습 방법』은 국어과 교수ㆍ학습의 주요 이론과 실제적인 적용 방법들을...,2007-09-01,2007-09-01 14:06:54
4,633,외국어로서의 한국문학교육,NULL,301,한국문화사,윤여탁,외국어로서의 한국어교육에서 한국 문학교육을 교육하는 이유와 방법들을 제시하고 있는 ...,2007-11-01,2007-11-01 15:23:39
5,428,하오빵 어린이 중국어 1 워크북,NULL,213,시사중국어사,김명화|이윤화,메인북에서 배운 중국어 표현들을 다시 한 번 점검해 볼 수 있도록 구성되어 있다. ...,2011-03-01,2011-03-01 13:31:53
...,...,...,...,...,...,...,...,...,...
84,579,국어과 교재 연구 및 지도법,49가지 교수학습 전략을 담은,282,사회평론아카데미,김주환|이순영|장은섭|이지영|최승식|정보 더 보기/감추기|김주환|이순영|장은섭|이지...,좋은 국어 수업을 만들기 위한핵심적인 교수 이론과 49가지 교수학습 전략‘국어과 교...,2024-09-01,2024-09-01 11:25:09
85,377,한국어능력시험(TOPIKII) 3급 도전부터 6급 합격까지 토픽 쓰기 노트,NULL,193,BOOKK(부크크),장수현,국제한국어문화학회가 기획한 『한국어능력시험(TOPIKⅡ) 3급 도전부터 6급 합격까...,2025-02-01,2025-02-01 12:41:32
86,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54
87,374,KBS 한국어능력시험 기출문제 23,NULL,191,형설출판사,KBS한국어진흥원,KBS한국어능력시험은 ‘국어기본법’ 관련 논의를 시작한 2002년부터 본격적인 준비...,2025-02-01,2025-02-01 14:30:42


- \[\] 은 안데 패턴에 포한할 문자셋을 지정합니다.

  book_title이 '패턴: (또는 )를 하나 이상 포함' 에 부합하는 행을 가져옵니다.

In [67]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[()]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
2,2727,동양 판타지 서유기 5권 (양장본),NULL,1324,현암사(전집),황근기,NULL,2004-07-01,2004-07-01 12:10:22
3,1546,대발견! 입체자연과학탐험 플러스 세트(전19권),NULL,733,키즈돔(KIZDOM),게오르그 할렌스레벤,NULL,2005-03-01,2005-03-01 16:38:35
4,2619,월간 디자인 (1년 정기구독),※사은품 미포함,1271,디자인하우스(잡지),마소팀,""" ** 해당상품은 업체배송상품입니다. 구매후 1주일안에 출판사에서 연락후 배송합니...",1976-10-01,1976-10-01 10:16:11
5,648,대학집주 (全),NULL,305,명문당,김혁제,NULL,1988-03-01,1988-03-01 13:40:26
...,...,...,...,...,...,...,...,...,...
197,1400,이기적 GTQ 인디자인 1급 (ver.CC),NULL,658,영진닷컴,김민숙,『이기적 인디자인 1급 (ver.CC)』 도서가 출간되었다. 본 도서는 GTQ 인디...,2024-10-01,2024-10-01 14:37:58
198,2588,포포포 매거진 POPOPO Magazine (반년간) : Issue No.09 [2...,NULL,1245,포포포,펫앤스토리,"“책은 읽기 싫지만, 내 책은 쓰고 싶어” 이 시대의 욕망을 출판 산업에서 발견합니...",2024-10-01,2024-10-01 09:18:38
199,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
200,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49


- \[\] 은 안에 패턴에 포한할 문자셋을 지정합니다.

  book_title이 '패턴: (하나 이상 포함하고, 임의의 문자열이 나오고,  )를 하나 이상 포함' 에 부합하는 행을 가져옵니다.

In [68]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[(]{1}.+[)]{1}')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
2,2727,동양 판타지 서유기 5권 (양장본),NULL,1324,현암사(전집),황근기,NULL,2004-07-01,2004-07-01 12:10:22
3,1546,대발견! 입체자연과학탐험 플러스 세트(전19권),NULL,733,키즈돔(KIZDOM),게오르그 할렌스레벤,NULL,2005-03-01,2005-03-01 16:38:35
4,2619,월간 디자인 (1년 정기구독),※사은품 미포함,1271,디자인하우스(잡지),마소팀,""" ** 해당상품은 업체배송상품입니다. 구매후 1주일안에 출판사에서 연락후 배송합니...",1976-10-01,1976-10-01 10:16:11
5,648,대학집주 (全),NULL,305,명문당,김혁제,NULL,1988-03-01,1988-03-01 13:40:26
...,...,...,...,...,...,...,...,...,...
197,1400,이기적 GTQ 인디자인 1급 (ver.CC),NULL,658,영진닷컴,김민숙,『이기적 인디자인 1급 (ver.CC)』 도서가 출간되었다. 본 도서는 GTQ 인디...,2024-10-01,2024-10-01 14:37:58
198,2588,포포포 매거진 POPOPO Magazine (반년간) : Issue No.09 [2...,NULL,1245,포포포,펫앤스토리,"“책은 읽기 싫지만, 내 책은 쓰고 싶어” 이 시대의 욕망을 출판 산업에서 발견합니...",2024-10-01,2024-10-01 09:18:38
199,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
200,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49


- [] 은 안데 패턴에 포한할 문자셋을 지정합니다.

  book_title이 '패턴: )하나 이상 포함하고, 임의의 문자열이 나오고,  (를 하나 이상 포함' 에 부합하는 행을 가져옵니다.

In [69]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[)]{1}.+[(]{1}')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2753,[웅진주니어] 마주보는 세계사(전8권)+한국사교실(전5권)세트,NULL,1336,웅진주니어(전집),편집부,NULL,2010-05-01,2010-05-01 11:43:32
2,356,Bricks Reading 250 (L1+L2+L3) SB (WB) 세트,NULL,182,Bricks(사회평론),David Cho,NULL,2014-11-01,2014-11-01 16:16:58
3,2694,신문이 보이고 뉴스가 들리는 재미있는 이야기 세트 (전40권) + 사은품증정(랜덤발송),교과학습 시사상식 논술대비까지 해결하는 초등학교 통합교과서,1312,가나출판사(전집),편집부,NULL,2017-06-01,2017-06-01 10:41:33
4,2668,선생님이 만든 좔좔 글읽기 2단계 세트(전3권)+사은품증정(랜덤발송),"교과서, 문학작품, 여러 글감을 통해 학습자의 읽기 속도를 고려한 다양한 자료 수록...",1302,다음생각,편집부,NULL,2017-10-01,2017-10-01 11:05:15
5,2726,(아람키즈) 아똥 그림책 (총 8종),"유아그림책,생활동화,창작동화,똥그림책,배변교육동화",1323,아람키즈(전집),황근기,NULL,2021-07-01,2021-07-01 10:27:43
6,351,해커스 지텔프 (G-TELP) 실전모의고사 청취 5회 Level 2 (레벨2),G-TELP 지텔프 시험 최신 기출유형 완벽 반영 | 군무원ㅣ경찰(순경)ㅣ해양경찰ㅣ...,179,해커스어학연구소,해커스 어학연구소,G-TELP 최신 기출유형 완벽 반영!한 권으로 공무원/경찰/군무원/소방/세무사/회...,2025-02-01,2025-02-01 13:01:52


In [70]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[()]{1}.+[()]{1}')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
2,2727,동양 판타지 서유기 5권 (양장본),NULL,1324,현암사(전집),황근기,NULL,2004-07-01,2004-07-01 12:10:22
3,1546,대발견! 입체자연과학탐험 플러스 세트(전19권),NULL,733,키즈돔(KIZDOM),게오르그 할렌스레벤,NULL,2005-03-01,2005-03-01 16:38:35
4,2619,월간 디자인 (1년 정기구독),※사은품 미포함,1271,디자인하우스(잡지),마소팀,""" ** 해당상품은 업체배송상품입니다. 구매후 1주일안에 출판사에서 연락후 배송합니...",1976-10-01,1976-10-01 10:16:11
5,648,대학집주 (全),NULL,305,명문당,김혁제,NULL,1988-03-01,1988-03-01 13:40:26
...,...,...,...,...,...,...,...,...,...
197,1400,이기적 GTQ 인디자인 1급 (ver.CC),NULL,658,영진닷컴,김민숙,『이기적 인디자인 1급 (ver.CC)』 도서가 출간되었다. 본 도서는 GTQ 인디...,2024-10-01,2024-10-01 14:37:58
198,2588,포포포 매거진 POPOPO Magazine (반년간) : Issue No.09 [2...,NULL,1245,포포포,펫앤스토리,"“책은 읽기 싫지만, 내 책은 쓰고 싶어” 이 시대의 욕망을 출판 산업에서 발견합니...",2024-10-01,2024-10-01 09:18:38
199,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
200,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49


- [] 은 안데 패턴에 포한할 문자셋을 지정합니다.

  publisher가 '텅 또는 사 를 포함' 에 부합하는 행을 가져옵니다.

In [71]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(publisher, '[텅사]')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2849,돼지가 철학에 빠진 날,NULL,1394,김영사,스티븐 로|오숙은,고등학교 국민윤리 시간에 끝없이 나열되는 철학자들의 이름과 요약정리된 이론에 진절머...,2001-07-01,2001-07-01 09:27:44
2,2361,에필로그,칼 세이건이 인류에게 남긴 마지막 메시지,1144,사이언스북스,칼 에드워드 세이건,유달리 대중들과 가깝게 지냈던 과학자 칼 세이건의 유작이 된 작품이다. 1996년 ...,2001-12-01,2001-12-01 14:50:06
3,2518,풀하우스,NULL,1204,사이언스북스,스티븐 제이 굴드,하버드대학 교수인 스티븐 제이 굴드는 그의 진화론을 책 한권에 꼭꼭 다져넣었다. 진...,2002-01-01,2002-01-01 14:36:21
4,132,알코올 및 약물 중독 질환을 위한 BRENDA 치료,약물 요법 및 심리 사회적 요법의 통합 치료,58,하나의학사,정영인,약물 요법도 수용하는 유연한 생물 심리 사회적 치료법을 이용하고자 하는 모든 치료자...,2002-03-01,2002-03-01 14:30:26
5,639,영어교육론,NULL,303,한국문화사,이광희,NULL,2002-08-01,2002-08-01 12:11:45
...,...,...,...,...,...,...,...,...,...
559,1355,2025 한 권으로 끝내는 요양보호사 국가자격시험 완전정복,NULL,637,사람과경영,요양보호사 학술연구회,"우리나라는 노인 인구 천 만 시대, 초 고령 사회 진입을 앞두고 노인 돌봄 인력 부...",2024-10-01,2024-10-01 10:37:07
560,2687,수학도둑 수학동화 8 9 10 전3권 세트,문구세트 증정,1310,서울문화사(전집),편집부,NULL,2024-10-01,2024-10-01 11:52:08
561,570,한국헌법론,NULL,277,박영사,허영,이번 판에서는 예년과 같이 2024년 선고된 헌법재판소의 판례와 대법원의 주요 판례...,2025-02-01,2025-02-01 16:19:14
562,374,KBS 한국어능력시험 기출문제 23,NULL,191,형설출판사,KBS한국어진흥원,KBS한국어능력시험은 ‘국어기본법’ 관련 논의를 시작한 2002년부터 본격적인 준비...,2025-02-01,2025-02-01 14:30:42


- [] 은 안데 패턴에 포한할 문자셋을 지정합니다.

  book_title이 '패턴: 0에서 9에 해당하는 문자에 하나 이상 포함' 에 부합하는 행을 가져옵니다.

In [72]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[0-9]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2830,명 1,역학의 맥,1385,갑을당,김상연,음양 오행을 바탕으로 전개되는 각종 이론과 명식작성법 및 해석방법 등을 오나전 초보...,2001-02-01,2001-02-01 09:41:49
2,1578,내 친구 쟈쟈 4,키드차이넷중국어,750,케이시,이재만|채홍범,어린이들을 위한 중국어 교재로 과학적인 교수법을 바탕으로 한 비주얼한 분위기와 입체...,2001-10-01,2001-10-01 14:08:57
3,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
4,2681,비룡소 클래식 베스트 20권세트/상품권1만,원작의 진지함을 고스란히 담은 세계 명작 완역 시리즈 비룡소 베스트 20선,1307,비룡소(전집),편집부,NULL,2002-10-01,2002-10-01 09:41:23
5,477,8822 중한사전,NULL,229,송산출판사,한학중,중국 언어학회 교수님들이 중국어 학습에 입문하는 이들이 위해 쉽게 접할 수 있도록 ...,2003-09-01,2003-09-01 09:55:08
...,...,...,...,...,...,...,...,...,...
870,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
871,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49
872,2672,도토리숲 시그림책 1~5권 세트,유아도서 증정-우리 집 하늘/콩알/보름달/바다거북이 장례식/겨울 들판,1303,도토리숲,편집부,NULL,2025-02-01,2025-02-01 12:42:08
873,834,오무라이스 잼잼 15,경이로운 일상음식 이야기,386,송송책방,조경규,올해도 오무라이스잼잼 15권이 찾아왔다. 훈훈하고 구수하고 넉넉하고 다정하게. 책장...,2025-02-01,2025-02-01 16:45:57


**문자 리스트**

|문자셋|설명|
|---|----|
|[:digit:]|숫자, [0-9]|
|[:lower:]|소문자, [a-z]|
|[:upper:]|대문자, [A-Z]|
|[:alpha:]|영문자, [a-zA-Z]|
|[:alnum:]|영문자와 숫자, [0-9a-zA-Z]|
|[:xdigit:]|16진숫자, [0-9a-fA-F]|
|[:punct:]|문장부호, [^[:alnum:][:cntrl:]]|
|[:space:]|공간문자(space, enter, tab)|
|[:blank:]|NULL 문자|

**.(dot) 연산자**: newline을 제외한 모든 문자

**PERL 문자 정의 연산자**
    
|연산자|설명|
|----|----|
|\d|숫자, [0-9], [[:digit:]]|
|\D|숫자가 아닌 모든 문자 [^0-9], [^[:digit:]]|
|\w|숫자와 영문자(under bar 포함] [0-9a-zA-Z_], [[:alnum:]_]|
|\W|숫자와 영문자, under bar가 아닌 모든 문자 <br/>[^0-9a-zA-Z_] , [^[:alnum:]_]|
|\s|공간문자[ \t\r], [[:space:]]|
|\S|비공간문자[^ \t\r], [^[:space:]]|


**[예제 10]** 문자를 나타내는 표현식을 사용해봅니다.

In [73]:
%%SQL

SELECT REGEXP_SUBSTR('abc', 'a.c') as R1,
    REGEXP_SUBSTR('abc', 'a.b') as R2,
    REGEXP_SUBSTR('abc', 'ab.') as R3,
    REGEXP_SUBSTR('abc', 'ac.') as R4
FROM dual

r1     abc
r2    NULL
r3     abc
r4    NULL
Name: 1, dtype: object

In [74]:
%%SQL

SELECT REGEXP_SUBSTR('abc', '[a-b][a-b][a-c]') as R1,
    REGEXP_SUBSTR('abc', '[a-b][a-b][a-b]') as R2,
    REGEXP_SUBSTR('abc', '[a-b][a-b][^a-c]') as R3,
    REGEXP_SUBSTR('abc', '[a-b][a-b][^a-b]') as R4
FROM dual

r1     abc
r2    NULL
r3    NULL
r4     abc
Name: 1, dtype: object

In [75]:
%%SQL

SELECT REGEXP_SUBSTR('aB1,', '[[:digit:]]') as R1,
    REGEXP_SUBSTR('aB1,', '[[:alpha:]]') as R2,
    REGEXP_SUBSTR('aB1,', '[[:lower:]]') as R3,
    REGEXP_SUBSTR('aB1,', '[[:upper:]]') as R4,
    REGEXP_SUBSTR('aB1,', '[[:alnum:]]') as R5,
    REGEXP_SUBSTR('aB1,', '[[:punct:]]') as R6
FROM dual

r1    1
r2    a
r3    a
r4    B
r5    a
r6    ,
Name: 1, dtype: object

**[예제11]** 사전에 정의된 문자셋과 PERL 문자 정의자를 써봅니다.

- '패턴: \[:digit:\]에 하나이상 포함'

  \[:digit:\] = \[0-9\]

In [76]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[[:digit:]]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2830,명 1,역학의 맥,1385,갑을당,김상연,음양 오행을 바탕으로 전개되는 각종 이론과 명식작성법 및 해석방법 등을 오나전 초보...,2001-02-01,2001-02-01 09:41:49
2,1578,내 친구 쟈쟈 4,키드차이넷중국어,750,케이시,이재만|채홍범,어린이들을 위한 중국어 교재로 과학적인 교수법을 바탕으로 한 비주얼한 분위기와 입체...,2001-10-01,2001-10-01 14:08:57
3,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
4,2681,비룡소 클래식 베스트 20권세트/상품권1만,원작의 진지함을 고스란히 담은 세계 명작 완역 시리즈 비룡소 베스트 20선,1307,비룡소(전집),편집부,NULL,2002-10-01,2002-10-01 09:41:23
5,477,8822 중한사전,NULL,229,송산출판사,한학중,중국 언어학회 교수님들이 중국어 학습에 입문하는 이들이 위해 쉽게 접할 수 있도록 ...,2003-09-01,2003-09-01 09:55:08
...,...,...,...,...,...,...,...,...,...
872,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
873,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49
874,2672,도토리숲 시그림책 1~5권 세트,유아도서 증정-우리 집 하늘/콩알/보름달/바다거북이 장례식/겨울 들판,1303,도토리숲,편집부,NULL,2025-02-01,2025-02-01 12:42:08
875,834,오무라이스 잼잼 15,경이로운 일상음식 이야기,386,송송책방,조경규,올해도 오무라이스잼잼 15권이 찾아왔다. 훈훈하고 구수하고 넉넉하고 다정하게. 책장...,2025-02-01,2025-02-01 16:45:57


- PERL 패턴 \d = [0-9] 에 대응

In [77]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '\d+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2830,명 1,역학의 맥,1385,갑을당,김상연,음양 오행을 바탕으로 전개되는 각종 이론과 명식작성법 및 해석방법 등을 오나전 초보...,2001-02-01,2001-02-01 09:41:49
2,1578,내 친구 쟈쟈 4,키드차이넷중국어,750,케이시,이재만|채홍범,어린이들을 위한 중국어 교재로 과학적인 교수법을 바탕으로 한 비주얼한 분위기와 입체...,2001-10-01,2001-10-01 14:08:57
3,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
4,2681,비룡소 클래식 베스트 20권세트/상품권1만,원작의 진지함을 고스란히 담은 세계 명작 완역 시리즈 비룡소 베스트 20선,1307,비룡소(전집),편집부,NULL,2002-10-01,2002-10-01 09:41:23
5,477,8822 중한사전,NULL,229,송산출판사,한학중,중국 언어학회 교수님들이 중국어 학습에 입문하는 이들이 위해 쉽게 접할 수 있도록 ...,2003-09-01,2003-09-01 09:55:08
...,...,...,...,...,...,...,...,...,...
872,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
873,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49
874,2672,도토리숲 시그림책 1~5권 세트,유아도서 증정-우리 집 하늘/콩알/보름달/바다거북이 장례식/겨울 들판,1303,도토리숲,편집부,NULL,2025-02-01,2025-02-01 12:42:08
875,834,오무라이스 잼잼 15,경이로운 일상음식 이야기,386,송송책방,조경규,올해도 오무라이스잼잼 15권이 찾아왔다. 훈훈하고 구수하고 넉넉하고 다정하게. 책장...,2025-02-01,2025-02-01 16:45:57


- '패턴: \[:lower:\]에 하나이상 포함'

  \[:lower:\] = \[a-z\]

In [78]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[[:lower:]]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3150,Object Oriented Perl,객체 지향 Perl,1514,인포북,이현주,"1980년대 후반 개발된 Perl은 주로 텍스트 파일을 검색하여, 실용적인 정보를 ...",2001-08-01,2001-08-01 16:37:16
2,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
3,652,Deutsch Fur Studenten 대학독일어,NULL,306,탐구당,전북대학교교양독일어교재편찬위,전북대학교교양독일어교재편찬위에서 집필한 대학독일어 교재이다.,1994-03-01,1994-03-01 11:08:38
4,3167,System Software,An Introduction to Systems Programming 3/E,1523,홍릉과학출판사,요게쉬 라헤자|주세페 보르게세|나다니엘 펠슨|이준호,이 책은 여러 형태의 시스템 소프트웨어의 설계와 구현에 대한 소개서이다. 이 책에서...,1999-03-01,1999-03-01 14:40:16
5,320,트루먼 쇼 the Truman Show,NULL,165,스크린영어사,이형식,영화를 보며 주어진 상황과 정황에 맞추어 실제로 살아 있는 현장감 있는 영어를 익힐...,2009-10-01,2009-10-01 12:07:31
...,...,...,...,...,...,...,...,...,...
167,3113,혼자 해도 프로 선생님처럼 잘 만드는 학교 수업 자료 with 캔바 Canva,현직 교사 5인이 알려주는 교육용 캔바를 활용한 에듀테크 수업의 모든 것,1499,한빛미디어,김태훈|김한나|김예슬|손민지|유수근,"선생님들을 위해, 선생님들이 집필한 에듀테크 캔바 활용 비법서캔바로 수업하면 학생 ...",2024-10-01,2024-10-01 11:30:03
168,1280,2025 영림원SystemEver ERP정보관리사 회계1·2급,NULL,608,도서출판배움,이진서|(주)영림원소프트랩,"ERP정보관리사 회계·1급/2급 대비 수험서이자, Cloud ERP를 직접 체험하게...",2025-02-01,2025-02-01 15:42:40
169,2986,20가지 템플릿으로 배우는 노션 Notion,"일정, 성과 관리, 포트폴리오, 아카이빙, 협업 문서까지 만들면서 배우는 노션",1454,시프트,전시진,NULL,2024-10-01,2024-10-01 16:10:16
170,1400,이기적 GTQ 인디자인 1급 (ver.CC),NULL,658,영진닷컴,김민숙,『이기적 인디자인 1급 (ver.CC)』 도서가 출간되었다. 본 도서는 GTQ 인디...,2024-10-01,2024-10-01 14:37:58


- '패턴: \[:upper:\]에 하나이상 포함'

  \[:upper:\] = \[A-Z\]

In [79]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[[:upper:]]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,3150,Object Oriented Perl,객체 지향 Perl,1514,인포북,이현주,"1980년대 후반 개발된 Perl은 주로 텍스트 파일을 검색하여, 실용적인 정보를 ...",2001-08-01,2001-08-01 16:37:16
2,132,알코올 및 약물 중독 질환을 위한 BRENDA 치료,약물 요법 및 심리 사회적 요법의 통합 치료,58,하나의학사,정영인,약물 요법도 수용하는 유연한 생물 심리 사회적 치료법을 이용하고자 하는 모든 치료자...,2002-03-01,2002-03-01 14:30:26
3,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
4,3148,ASP웹프로그래밍,NULL,1512,내하출판사,이교운|정경환,"이 책은 아직 ASP를 전혀 접해보지 못한 독자를 위해서 ASP를 이해하고, ASP...",2003-08-01,2003-08-01 14:32:23
5,980,PD가 말하는 PD,NULL,449,부키,김민식,"PD라는 이름으로 묶여 있을 뿐, 하는 일도 성격도 천차만별인 PD라는 직업에 대해...",2003-12-01,2003-12-01 16:49:23
...,...,...,...,...,...,...,...,...,...
393,1508,UNION PSAT&LEET 법률문제 심층분석,NULL,705,인해,백현관,대표 문제들을 유형별로 분류하고 단계별 풀이과정을 통해 법률문제를 빠르고 쉽게 접근...,2025-02-01,2025-02-01 16:38:40
394,1046,K-방산에 투자하라,"마지막 남은 투자 기회 트럼프 2기, 방산으로 돈이 모인다!",476,위즈덤하우스,김민석,"트럼프 2.0 시대 최고 수혜주, K-방산에 주목하라!기술부터 제품, 시장, 산업 ...",2025-02-01,2025-02-01 09:20:11
395,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
396,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49


- '패턴: \[:alpha:\]에 하나이상 포함'

  \[:alpha:\] = \[a-zA-Z\]

In [80]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[[:alpha:]]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2164,황소 아저씨,NULL,1033,길벗어린이,권정생|정승각,추운 겨울밤에 황소 아저씨와 새앙쥐 남매들이 나누는 가슴 따뜻한 이야기. 엄마가 돌...,2001-01-01,2001-01-01 10:22:03
2,2830,명 1,역학의 맥,1385,갑을당,김상연,음양 오행을 바탕으로 전개되는 각종 이론과 명식작성법 및 해석방법 등을 오나전 초보...,2001-02-01,2001-02-01 09:41:49
3,2377,이승만 다시 보기,우리가 버린 건국의 아버지,1150,기파랑,인보길,이승만 다시 바라보기최근 이승만연구소를 발족시킨 인터넷신문 뉴데일리가 연구총서 제1...,2001-03-01,2001-03-01 16:53:22
4,2158,뭐하니?,NULL,1030,길벗어린이,유문조|최민오,'까꿍놀이'를 주제로 한 아기 그림책입니다. 책을 펼치면 구부정하게 등을 구부리고 ...,2001-05-01,2001-05-01 15:29:39
5,2849,돼지가 철학에 빠진 날,NULL,1394,김영사,스티븐 로|오숙은,고등학교 국민윤리 시간에 끝없이 나열되는 철학자들의 이름과 요약정리된 이론에 진절머...,2001-07-01,2001-07-01 09:27:44
...,...,...,...,...,...,...,...,...,...
2592,2672,도토리숲 시그림책 1~5권 세트,유아도서 증정-우리 집 하늘/콩알/보름달/바다거북이 장례식/겨울 들판,1303,도토리숲,편집부,NULL,2025-02-01,2025-02-01 12:42:08
2593,834,오무라이스 잼잼 15,경이로운 일상음식 이야기,386,송송책방,조경규,올해도 오무라이스잼잼 15권이 찾아왔다. 훈훈하고 구수하고 넉넉하고 다정하게. 책장...,2025-02-01,2025-02-01 16:45:57
2594,381,"2025 한국어능력시험 TOPIK Ⅰ(토픽 1) 한 번에 통과하기+온라인 시험, 무...",NULL,193,시대고시기획 시대교육,한국어능력시험연구회|임준,무료 강의 +핵심 이론(어휘·문법) + 모의고사 + 한·영·중 해설 + 단어장실제 ...,2025-02-01,2025-02-01 14:49:55
2595,1447,나합격 금속재료기능장 필기+실기,NULL,679,삼원북스,나합격콘텐츠연구소,필기시험대비 이론&기출 수록. 최신 필답형 복원문제.,2025-02-01,2025-02-01 16:10:56


- '패턴: \[:punct:\]에 하나이상 포함'

  \[:punct:\] = 문장부호

In [81]:
%%SQL

SELECT * FROM book WHERE REGEXP_LIKE(book_title, '[[:punct:]]+')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2158,뭐하니?,NULL,1030,길벗어린이,유문조|최민오,'까꿍놀이'를 주제로 한 아기 그림책입니다. 책을 펼치면 구부정하게 등을 구부리고 ...,2001-05-01,2001-05-01 15:29:39
2,630,〈그리고 두려운 것〉에 대한 인문학적 고찰,NULL,301,전남대학교출판문화원,이준학,NULL,2001-11-01,2001-11-01 09:02:25
3,2261,"선악의 저편, 도덕의 계보",NULL,1095,책세상,프리드리히 니체|김정현,니체 사거 100주년을 맞아 책세상 출판사에서 내고 있는 니체전집 중 한권이다. 책...,2002-02-01,2002-02-01 10:17:13
4,3145,Visual C++.Net Programming Bible (전3권),NULL,1509,삼양미디어,박광우|강우경|진용철,"저자가 생각하고 있는, 프로그래밍에 입문하는 독자들이 택할 수 있는 두가지 길은 ...",2002-08-01,2002-08-01 09:50:47
5,2681,비룡소 클래식 베스트 20권세트/상품권1만,원작의 진지함을 고스란히 담은 세계 명작 완역 시리즈 비룡소 베스트 20선,1307,비룡소(전집),편집부,NULL,2002-10-01,2002-10-01 09:41:23
...,...,...,...,...,...,...,...,...,...
643,1293,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET,제27회 시험대비,615,시대고시기획 시대교육,시대에듀 경비지도사 교수진,『2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET』의 ...,2025-02-01,2025-02-01 14:20:40
644,1520,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...,NULL,715,시대고시기획 시대교육,시대PSAT연구소,7급 및 민간경력자 PSAT 시험을 준비하는 수험생들에게 도움이 되고자 『2025 ...,2025-02-01,2025-02-01 11:00:49
645,2672,도토리숲 시그림책 1~5권 세트,유아도서 증정-우리 집 하늘/콩알/보름달/바다거북이 장례식/겨울 들판,1303,도토리숲,편집부,NULL,2025-02-01,2025-02-01 12:42:08
646,381,"2025 한국어능력시험 TOPIK Ⅰ(토픽 1) 한 번에 통과하기+온라인 시험, 무...",NULL,193,시대고시기획 시대교육,한국어능력시험연구회|임준,무료 강의 +핵심 이론(어휘·문법) + 모의고사 + 한·영·중 해설 + 단어장실제 ...,2025-02-01,2025-02-01 14:49:55


- 문장 부호를 공백으로 치환

In [82]:
%%SQL

SELECT REGEXP_REPLACE(book_title, '[[:punct:]]+'), book_title FROM book WHERE REGEXP_LIKE(book_title, '[[:punct:]]+')

,"REGEXP_REPLACE(BOOK_TITLE,'[[:PUNCT:]]+')",book_title
1,뭐하니,뭐하니?
2,그리고 두려운 것에 대한 인문학적 고찰,〈그리고 두려운 것〉에 대한 인문학적 고찰
3,선악의 저편 도덕의 계보,"선악의 저편, 도덕의 계보"
4,Visual CNet Programming Bible 전3권,Visual C++.Net Programming Bible (전3권)
5,비룡소 클래식 베스트 20권세트상품권1만,비룡소 클래식 베스트 20권세트/상품권1만
...,...,...
643,2025 시대에듀 경비지도사 1차 기본서 2종 법학개론민간경비론 SET,2025 시대에듀 경비지도사 1차 기본서 2종 (법학개론+민간경비론) SET
644,2025 시대에듀 7급민간경력자 PSAT 전과목 단기완성필수기출 300제언어논리자료...,2025 시대에듀 7급/민간경력자 PSAT 전과목 단기완성+필수기출 300제(언어논...
645,도토리숲 시그림책 15권 세트,도토리숲 시그림책 1~5권 세트
646,2025 한국어능력시험 TOPIK Ⅰ토픽 1 한 번에 통과하기온라인 시험 무료 강의,"2025 한국어능력시험 TOPIK Ⅰ(토픽 1) 한 번에 통과하기+온라인 시험, 무..."


**특수문자**

|연산자|설명|
|----|----|
|\t|tab|
|\n|newline|
|\r|return|
|\b|backspace|

**앵커**

|연산자|설명|
|----|-----|
|^|문자열의 시작|
|$|문자열의 끝|


**수량사(Qualifier)**

<table>
  <tr>
    <th colspan="2">수량연산자</th><th>설명</th>
  </tr>
  <tr>
    <th>탐욕적 방식</th><th>비탐욕적 방식</th><th>설명</th>
  </tr>
  <tr>
    <td>?</td><td>?</td><td>0회 또는 1회 일치</td>
  </tr>
  <tr>
    <td>*</td><td>*?</td><td>0회 또는 그 이상 일치</td>
  </tr>
  <tr>
    <td>+</td><td>+?</td><td>1회 또는 그 이상 일치</td>
  </tr>
  <tr>
    <td>{m}</td><td>{m}?</td><td>m회 일치</td>
  </tr>
  <tr>
    <td>{m,}</td><td>{m,}?</td><td>최소 m회 이상 일치</td>
  </tr>
  <tr>
    <td>{,m }</td><td>{,m}?</td><td>최대 m회 일치</td>
  </tr>
  <tr>
    <td>{m,n}</td><td>{m,n}?</td><td>최소 m회 최대 n회 일치</td>
  </tr>
</table>


**특수 연산자**

|연산자|설명|
|-----|-----|
|\||OR 연산, 정규표현식1|정규표현식2|
|\\|이스케이프 문자: 연산자를 일반 문자로 취급|

**[예제 12]** 앵커기능을 확인합니다.

In [83]:
%%SQL
SELECT REGEXP_SUBSTR('CABBBB', '^AB+?'), REGEXP_SUBSTR('CABBBB', 'AB+') FROM DUAL

REGEXP_SUBSTR('CABBBB','^AB+?')     NULL
REGEXP_SUBSTR('CABBBB','AB+')      ABBBB
Name: 1, dtype: object

In [84]:
%%SQL
SELECT REGEXP_SUBSTR('ABBBBC', 'AB+?$'), REGEXP_SUBSTR('ABBBBC', 'AB+') FROM DUAL

REGEXP_SUBSTR('ABBBBC','AB+?$')     NULL
REGEXP_SUBSTR('ABBBBC','AB+')      ABBBB
Name: 1, dtype: object

**[예제 13]** 수량 한정사아 비탐욕적은 정규 표현식에 해당하는 문자열의 길이를 작게, 탐욕적은 되도록이면 길게 일치시키는 방법에 대해 알아봅니다.

In [85]:
%%SQL
SELECT REGEXP_SUBSTR('ABBBBC', 'AB{1}'), REGEXP_SUBSTR('ABBBBC', 'AB{2}') FROM DUAL

REGEXP_SUBSTR('ABBBBC','AB{1}')     AB
REGEXP_SUBSTR('ABBBBC','AB{2}')    ABB
Name: 1, dtype: object

In [86]:
%%SQL
SELECT REGEXP_SUBSTR('ABBBBC', '(AB){1}'), REGEXP_SUBSTR('ABBBBC', '(AB){2}') FROM DUAL

REGEXP_SUBSTR('ABBBBC','(AB){1}')      AB
REGEXP_SUBSTR('ABBBBC','(AB){2}')    NULL
Name: 1, dtype: object

In [87]:
%%SQL

SELECT REGEXP_SUBSTR('ABBBB', 'AB+?'), REGEXP_SUBSTR('ABBBB', 'AB+') FROM DUAL

REGEXP_SUBSTR('ABBBB','AB+?')       AB
REGEXP_SUBSTR('ABBBB','AB+')     ABBBB
Name: 1, dtype: object

In [88]:
%%SQL

SELECT REGEXP_SUBSTR('ABBBBC', 'AB+?C'), REGEXP_SUBSTR('ABBBBC', 'AB+C') FROM DUAL

REGEXP_SUBSTR('ABBBBC','AB+?C')    ABBBBC
REGEXP_SUBSTR('ABBBBC','AB+C')     ABBBBC
Name: 1, dtype: object

**[예제 13]** |는 정규 표현식에서 "또는(or)"을 나타낼 수 있습니다.

In [89]:
%%SQL

SELECT REGEXP_SUBSTR('ABBBBC', 'AB$'), REGEXP_SUBSTR('ABBBBC', '^AB'), REGEXP_SUBSTR('ABBBBC', 'AB$|^AB') FROM DUAL

REGEXP_SUBSTR('ABBBBC','AB$')        NULL
REGEXP_SUBSTR('ABBBBC','^AB')          AB
REGEXP_SUBSTR('ABBBBC','AB$|^AB')      AB
Name: 1, dtype: object

**[예제 14]** \는 정규 표현식에서 특수 문자를 일반 문자로 나타낼 수 있게 합니다.

In [90]:
%%SQL

SELECT REGEXP_SUBSTR('\\', '\+?') FROM DUAL

REGEXP_SUBSTR('\\','\+?')    NULL
Name: 1, dtype: object

In [91]:
%%SQL

SELECT REGEXP_SUBSTR('\\', '\\+?') FROM DUAL

REGEXP_SUBSTR('\\','\\+?')    \
Name: 1, dtype: object